In [3]:
from dotenv import load_dotenv
import os
import pprint as pp
from opensearchpy import OpenSearch
from opensearchpy import helpers

load_dotenv()

OPENSEARCH_USER = os.getenv("OPENSEARCH_USER")
OPENSEARCH_PASSWORD = os.getenv("OPENSEARCH_PASSWORD")
OPENSEARCH_HOST = os.getenv("OPENSEARCH_HOST")
OPENSEARCH_PORT = os.getenv("OPENSEARCH_PORT")

index_name = OPENSEARCH_USER + '_project'

In [4]:
index_name = OPENSEARCH_USER + '_project'
# Create the client with SSL/TLS enabled, but hostname verification disabled.
client = OpenSearch(
    hosts = [{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}],
    http_compress = True, # enables gzip compression for request bodies
    http_auth = (OPENSEARCH_USER, OPENSEARCH_PASSWORD),
    use_ssl = True,
    url_prefix = 'opensearch_v3',
    verify_certs = False,
    ssl_assert_hostname = False,
    ssl_show_warn = False
)


if client.indices.exists(index=index_name):

    resp = client.indices.open(index=index_name)
    print(resp)

    print("--- INDEX SETTINGS")
    settings = client.indices.get_settings(index=index_name)
    pp.pprint(settings)

    print("--- INDEX MAPPINGS")
    mappings = client.indices.get_mapping(index=index_name)
    pp.pprint(mappings)

    print("--- INDEX #DOCs")
    print(client.count(index=index_name))
else:
    print("Index does not exist.")


{'acknowledged': True, 'shards_acknowledged': True}
--- INDEX SETTINGS
{'uservl07_project': {'settings': {'index': {'creation_date': '1774973269602',
                                             'knn': 'true',
                                             'knn.derived_source': {'enabled': 'true'},
                                             'number_of_replicas': '0',
                                             'number_of_shards': '4',
                                             'provided_name': 'uservl07_project',
                                             'refresh_interval': '1s',
                                             'replication': {'type': 'DOCUMENT'},
                                             'similarity': {'bm25-0-75': {'b': '0.75',
                                                                          'k1': '0.0',
                                                                          'type': 'BM25'},
                                                            

In [5]:
from openai import OpenAI

BASE_URL = os.getenv("BASE_URL")
API_KEY = os.getenv("API_KEY")
MODEL = os.getenv("MODEL")

print(repr(BASE_URL), repr(API_KEY), repr(MODEL))
openai_client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
#NOTE: the model parameter is case sensitive, and must be exactly "google/gemma-4-31B-it" for the API to find it

reponse = openai_client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of Portugal?"}
    ]
)
print(reponse.choices[0].message.content)

'https://api.novasearch.org/gemma4/v1' 'nova-vl' 'google/gemma-4-31B-it'
The capital of Portugal is Lisbon.


In [6]:
import base64
from IPython.display import Image, display

#openai_client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
image_url = "lisbon.jpg"
with open(image_url, "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()
display(Image(url=image_url, width=400))
response = openai_client.chat.completions.create(
model=MODEL,
messages=[
        {"role": "system", "content": "You are a helpful visual assistant."},
        {"role": "user", "content": [
            {"type": "image_url", "image_url":
                {"url": f"data:image/jpeg;base64,{image_b64}"}},
            {"type": "text", "text": "What is happening in this image?"}
        ]}
    ]
)
print(response.choices[0].message.content)

KeyboardInterrupt: 

In [ ]:
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import aiohttp


ds = load_dataset("HuggingFaceM4/COCO", split="validation",
                  trust_remote_code=True,
                  storage_options={"client_kwargs": {"timeout": aiohttp.ClientTimeout(total=36000)}}
                  )
sbert_model = SentenceTransformer('all-mpnet-base-v2')
bge_model = SentenceTransformer('BAAI/bge-small-en-v1.5')

##NOTE: review when index is back online, to check that the field names in the query match those in the index mapping
def retrieve_from_query(text_query, index_name=index_name, modality='BM25', top_k=3):
    if modality == 'BM25':
        query_body = {
            "size": top_k,
            "query": {
                "match": {
                    "caption": text_query
                }
            }
        }
    elif modality == 'SBERT':
        query_embedding = sbert_model.encode(text_query)
        query_body = {
            "size": top_k,
            "query": {
                "knn": {
                    "caption_vector": {
                        "vector": query_embedding,
                        "k": top_k
                    }
                }
            }
        }
    elif modality == 'BGE':
        query_embedding = bge_model.encode(text_query)
        query_body = {
            "size": top_k,
            "query": {
                "knn": {
                    "caption_vector_bge": {
                        "vector": query_embedding,
                        "k": top_k
                    }
                }
            }
        }
    else:
        raise ValueError(f"Unsupported modality: {modality}")
    
    response = client.search(index=index_name, body=query_body)
    return response['hits']['hits']

def generate_llm_answer(text_query, top_response, original_caption = False):
    #pass the top retrieved image and the original question to the LVLM via the vLLM endpoint to produce an answer.
    image_id = top_response['_id']['image_id'] #NOTE: check the field name when index is back online
    image = ds[int(image_id)]['image'] #NOTE: check the field name when loading the dataset
    image_b64 = base64.b64encode(image).decode()
    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful visual assistant."},
            {"role": "user", "content": [
                {"type": "image_url", "image_url":
                    {"url": f"data:image/jpeg;base64,{image_b64}"}},
                {"type": "text", "text": text_query if not original_caption else f"{text_query} The original caption for this image is: {top_response['_source']['caption']}"}
            ]}
        ]
    )
    return response.choices[0].message.content